# Aadhaar KYC Face Matching Pipeline - SageMaker

Exact same pipeline running locally on SageMaker with HuggingFace models.

- **Instance:** `ml.g5.xlarge` (A10G 24GB) or any GPU with 20GB+ VRAM
- **Volume:** 100GB (models ~20GB total)
- **No Ollama** — VLM runs directly via HuggingFace transformers

## Step 1: Clone repo

In [ ]:
# Already cloned at /home/pragati/mlvprasad/GENAI_AADHAR
%cd /home/pragati/mlvprasad/GENAI_AADHAR
!git pull origin main

## Step 2: Check GPU

In [ ]:
!nvidia-smi

## Step 3: Install dependencies

In [ ]:
import sys
# Install order matters — do NOT rearrange
# Using sys.executable ensures packages install into the Jupyter kernel's Python

# PyTorch with CUDA (use latest compatible, not pinned version)
!{sys.executable} -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!{sys.executable} -m pip install numpy==1.26.4
!{sys.executable} -m pip install onnxruntime-gpu==1.19.2 || {sys.executable} -m pip install onnxruntime==1.19.2
!{sys.executable} -m pip install basicsr==1.4.2
!{sys.executable} -m pip install realesrgan==0.3.0
!{sys.executable} -m pip install insightface==0.7.3
!{sys.executable} -m pip install opencv-python==4.10.0.84 Pillow==10.4.0 PyYAML==6.0.2 requests==2.32.3 tqdm==4.66.4 PyMuPDF==1.24.5
!{sys.executable} -m pip install transformers accelerate qwen-vl-utils pytest==8.3.2

In [ ]:
# Verify GPU
import torch
print(f"PyTorch CUDA: {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'})")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

import onnxruntime
print(f"ONNX Runtime: {onnxruntime.get_available_providers()}")

## Step 4: Set local Qwen2.5-VL-7B-Instruct model path

Point `config.yaml` to your locally downloaded model. Update the path below to match your SageMaker folder.

In [ ]:
import yaml

# ── Your local Qwen2.5-VL-7B-Instruct path on SageMaker ──
LOCAL_MODEL_PATH = "/home/pragati/My Models/Qwen2.5-VL-7B-Instruct"

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

config["vlm_guard"]["model_path"] = LOCAL_MODEL_PATH
config["vlm_guard"]["enabled"] = True

with open("config.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"VLM model path set to: {LOCAL_MODEL_PATH}")
print("Verify model files exist:")
import os
if os.path.isdir(LOCAL_MODEL_PATH):
    json_files = [f for f in os.listdir(LOCAL_MODEL_PATH) if f.endswith('.json')]
    print(f"  Found {len(json_files)} config files, {len(os.listdir(LOCAL_MODEL_PATH))} total files")
else:
    print(f"  WARNING: {LOCAL_MODEL_PATH} not found — update LOCAL_MODEL_PATH above")

## Step 5: Download pipeline model weights (InsightFace + Real-ESRGAN)

In [ ]:
!python scripts/download_models.py

In [ ]:
# Verify all models
!echo "=== InsightFace ==="  && ls models/insightface/models/buffalo_l/
!echo "=== Real-ESRGAN ===" && ls models/realesrgan/
!echo "=== Qwen2.5-VL ==="  && ls models/qwen2.5-vl-7b-instruct/*.json | head -5

## Step 6: Upload test images

In [ ]:
!mkdir -p FILES/AADHAR FILES/SELFIE

# Copy from S3:
# !aws s3 cp s3://your-bucket/test-data/AADHAR/ FILES/AADHAR/ --recursive
# !aws s3 cp s3://your-bucket/test-data/SELFIE/ FILES/SELFIE/ --recursive

# Or upload via Jupyter file browser (drag & drop)

!echo "Aadhaar files:" && ls FILES/AADHAR/
!echo "Selfie files:"  && ls FILES/SELFIE/

## Step 7: Run tests

In [ ]:
!python -m pytest tests/ -v -m "not integration" --tb=short 2>&1 | tail -5

## Step 8: Run pipeline

In [ ]:
# Single pair
!python main.py --aadhaar FILES/AADHAR/AADHAR001.jpg --selfie FILES/SELFIE/USER_01.jpg --verbose

In [ ]:
# Batch mode - all 90 pairs
!python main.py --batch pairs.csv --verbose

In [ ]:
# Check results
import os
latest = sorted([d for d in os.listdir('logs') if d.startswith('batch_')])[-1]
!cat logs/{latest}/summary.txt

## Step 9: Use as Python library

In [ ]:
from utils.config_loader import load_config
from pipeline.orchestrator import KYCPipelineOrchestrator

config = load_config('config.yaml')
pipeline = KYCPipelineOrchestrator(config)
pipeline.load()

aadhaar_bytes = open('FILES/AADHAR/AADHAR001.jpg', 'rb').read()
selfie_bytes  = open('FILES/SELFIE/USER_01.jpg', 'rb').read()

result = pipeline.run(aadhaar_bytes, selfie_bytes)

print(f"Match:      {result.match}")
print(f"Confidence: {result.confidence_pct}%")
print(f"Cosine:     {result.cosine_score:.4f}")
print(f"Fused:      {result.fused_score:.4f}")
print(f"VLM:        {result.vlm_reasoning}")
print(f"Aadhaar:    {result.aadhaar_gender} age {result.aadhaar_age}")
print(f"Selfie:     {result.selfie_gender} age {result.selfie_age}")